# Chapter 3: Systems of Linear Equations

**Every** machine learning model ultimately solves (or optimises) a system of equations.
Linear regression finds $\hat{x}$ minimising $\|Ax - b\|^2$; neural network training solves a gradient equation.

### Example: 2-equation system
$$2x_1 + x_2 = 5$$
$$x_1 - x_2 = 1$$

In matrix form $Ax = b$:
$$\underbrace{\begin{bmatrix}2 & 1 \\ 1 & -1\end{bmatrix}}_{A} \underbrace{\begin{bmatrix}x_1 \\ x_2\end{bmatrix}}_{x} = \underbrace{\begin{bmatrix}5 \\ 1\end{bmatrix}}_{b}$$

## Learning Objectives
- Translate a system of equations into matrix form
- Understand the three possible solution cases geometrically
- Perform Gaussian elimination by hand (then let NumPy do it)
- Solve overdetermined systems via least squares


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg as sla

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)


## From Equations to Matrix Form

Given a system:
$$a_{11}x_1 + a_{12}x_2 + \cdots + a_{1n}x_n = b_1$$
$$\vdots$$
$$a_{m1}x_1 + a_{m2}x_2 + \cdots + a_{mn}x_n = b_m$$

We write it compactly as $Ax = b$ where:
- $A \in \mathbb{R}^{m \times n}$ is the **coefficient matrix**
- $x \in \mathbb{R}^n$ is the **unknown vector**
- $b \in \mathbb{R}^m$ is the **right-hand side**


In [ ]:
# 2x1 + x2 = 5
# x1 - x2  = 1
A = np.array([[2,  1],
              [1, -1]], dtype=float)
b = np.array([5, 1], dtype=float)

x_known = np.array([2.0, 1.0])  # proposed solution
print("Verify A @ x_known == b?", np.allclose(A @ x_known, b))
print("Residual:", A @ x_known - b)


## Geometric Interpretation (2D)

Each equation $a_1 x_1 + a_2 x_2 = b$ is a **line** in the plane.

Three cases for two equations in two unknowns:

1. **Unique solution** — lines intersect at one point
2. **No solution** — lines are parallel (inconsistent)
3. **Infinite solutions** — lines are identical (dependent)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x_vals = np.linspace(-2, 5, 200)

# Case 1: Unique solution
ax = axes[0]
ax.plot(x_vals, 5 - 2*x_vals, 'b', lw=2, label=r'$2x+y=5$')
ax.plot(x_vals, x_vals - 1,   'r', lw=2, label=r'$x-y=1$')
ax.plot(2, 1, 'ko', ms=10, zorder=5, label='Solution (2,1)')
ax.set(xlim=(-1,4), ylim=(-2,5), title='Case 1: Unique Solution')
ax.legend(fontsize=8); ax.set_aspect('equal')

# Case 2: No solution (parallel)
ax = axes[1]
ax.plot(x_vals, 3 - x_vals,   'b', lw=2, label=r'$x+y=3$')
ax.plot(x_vals, 5 - x_vals,   'r', lw=2, label=r'$x+y=5$')
ax.set(xlim=(-1,5), ylim=(-1,6), title='Case 2: No Solution (Parallel)')
ax.legend(fontsize=8); ax.set_aspect('equal')

# Case 3: Infinite solutions (same line)
ax = axes[2]
ax.plot(x_vals, 3 - x_vals,       'b', lw=4, alpha=0.5, label=r'$x+y=3$')
ax.plot(x_vals, (6 - 2*x_vals)/2, 'r--', lw=2,         label=r'$2x+2y=6$')
ax.set(xlim=(-1,5), ylim=(-1,5), title='Case 3: Infinite Solutions')
ax.legend(fontsize=8); ax.set_aspect('equal')

plt.suptitle('Three Cases for Systems of Linear Equations', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## Gaussian Elimination

Transform the **augmented matrix** $[A \mid b]$ to **row echelon form** using:
1. **Row swap**: $R_i \leftrightarrow R_j$
2. **Row scale**: $R_i \leftarrow \alpha R_i$
3. **Row addition**: $R_i \leftarrow R_i + \alpha R_j$

Then **back-substitute** from the last equation upward.


In [ ]:
# Solve:  2x + y - z = 8
#          -3x - y + 2z = -11
#          -2x + y + 2z = -3

A = np.array([[ 2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]], dtype=float)
b = np.array([8, -11, -3], dtype=float)

# Build augmented matrix
aug = np.hstack([A, b.reshape(-1,1)])
print("Augmented [A|b]:\n", aug)

# Step 1: eliminate x from rows 2 and 3
aug[1] = aug[1] - (aug[1,0]/aug[0,0]) * aug[0]
aug[2] = aug[2] - (aug[2,0]/aug[0,0]) * aug[0]
print("\nAfter eliminating column 0:\n", aug)

# Step 2: eliminate y from row 3
aug[2] = aug[2] - (aug[2,1]/aug[1,1]) * aug[1]
print("\nRow echelon form:\n", aug)

# Back substitution
x = np.zeros(3)
x[2] = aug[2,3] / aug[2,2]
x[1] = (aug[1,3] - aug[1,2]*x[2]) / aug[1,1]
x[0] = (aug[0,3] - aug[0,1]*x[1] - aug[0,2]*x[2]) / aug[0,0]
print(f"\nSolution: x={x[0]:.2f}, y={x[1]:.2f}, z={x[2]:.2f}")
print("Verify A@x == b?", np.allclose(A @ x, b))


## Solving with NumPy

`np.linalg.solve(A, b)` uses **LAPACK/BLAS** routines (LU decomposition internally).
It is **always** faster and more numerically stable than computing the inverse explicitly.

The **condition number** $\kappa(A)$ measures sensitivity to perturbations:
- $\kappa \approx 1$ → well-conditioned (safe)
- $\kappa \gg 1$ → ill-conditioned (results may be inaccurate)


In [ ]:
import time

A = np.array([[ 2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]], dtype=float)
b = np.array([8., -11., -3.])

x = np.linalg.solve(A, b)
print("Solution:", x)
print("Condition number:", np.linalg.cond(A))

# Timing comparison on larger system
n = 500
A_big = np.random.randn(n, n)
b_big = np.random.randn(n)

t0 = time.perf_counter()
for _ in range(20): np.linalg.solve(A_big, b_big)
t_solve = (time.perf_counter() - t0) / 20

t0 = time.perf_counter()
for _ in range(20): np.linalg.inv(A_big) @ b_big
t_inv = (time.perf_counter() - t0) / 20

print(f"\n500×500 system — solve: {t_solve*1e3:.2f} ms | inv@b: {t_inv*1e3:.2f} ms")
print(f"solve is {t_inv/t_solve:.1f}× faster")


## Solution Types and Rank

Let $r = \text{rank}(A)$ and $n$ = number of unknowns.

| Condition | Solutions |
|---|---|
| $r = n$ and $\det(A) \neq 0$ | **Unique** solution |
| $\text{rank}([A\|b]) > \text{rank}(A)$ | **No** solution (inconsistent) |
| $r < n$ | **Infinitely many** solutions |


In [ ]:
# Case 1: Unique
A1 = np.array([[2.,1.],[1.,-1.]]); b1 = np.array([5.,1.])
print("Case 1 rank:", np.linalg.matrix_rank(A1),
      "| solution:", np.linalg.solve(A1, b1))

# Case 2: No solution
A2 = np.array([[1.,1.],[1.,1.]]); b2 = np.array([3.,5.])
aug2 = np.hstack([A2, b2.reshape(-1,1)])
print("\nCase 2 rank(A):", np.linalg.matrix_rank(A2),
      "| rank([A|b]):", np.linalg.matrix_rank(aug2))
try:
    np.linalg.solve(A2, b2)
except np.linalg.LinAlgError as e:
    print("  Error:", e)

# Case 3: Infinite solutions
A3 = np.array([[1.,2.],[2.,4.]]); b3 = np.array([3.,6.])
print("\nCase 3 rank(A):", np.linalg.matrix_rank(A3),
      "| particular solution:", np.linalg.lstsq(A3, b3, rcond=None)[0])


## Overdetermined Systems & Least Squares

When $m > n$ (more equations than unknowns), an exact solution rarely exists.
Instead, we **minimise the residual**:

$$\hat{x} = \arg\min_x \|Ax - b\|^2$$

The closed-form solution uses the **normal equations**:

$$\hat{x} = (A^T A)^{-1} A^T b$$

In practice: use `np.linalg.lstsq(A, b)` (more stable via QR/SVD internally).


In [ ]:
np.random.seed(0)
n_pts = 50
x_data = np.linspace(0, 4, n_pts)
y_data = 2.5 * x_data + 1.0 + np.random.randn(n_pts) * 0.8

# Augment with bias column
A_ls = np.column_stack([x_data, np.ones(n_pts)])
x_hat, residuals, rank, sv = np.linalg.lstsq(A_ls, y_data, rcond=None)
slope, intercept = x_hat
print(f"Fitted line: y = {slope:.3f}x + {intercept:.3f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
y_fit = A_ls @ x_hat
ax1.scatter(x_data, y_data, alpha=0.6, label='Data')
ax1.plot(x_data, y_fit, 'r', lw=2, label=f'Fit: {slope:.2f}x+{intercept:.2f}')
ax1.set(title='Least Squares Fit', xlabel='x', ylabel='y'); ax1.legend()

ax2.stem(x_data, y_data - y_fit, markerfmt='C0o', linefmt='C0-', basefmt='k-')
ax2.axhline(0, color='r', lw=2); ax2.set(title='Residuals', xlabel='x', ylabel='residual')
plt.tight_layout(); plt.show()


## Polynomial Fitting & the Vandermonde Matrix

Fitting a degree-$d$ polynomial $\sum_{k=0}^d c_k x^k$ is linear in coefficients $c_k$.
The design matrix is the **Vandermonde matrix**:

$$V = \begin{bmatrix} 1 & x_1 & x_1^2 & \cdots & x_1^d \\ \vdots & & & & \vdots \\ 1 & x_n & x_n^2 & \cdots & x_n^d \end{bmatrix}$$

> Caution: condition number explodes with high degree → numerical instability.


In [ ]:
np.random.seed(1)
x_p = np.linspace(0, 2*np.pi, 30)
y_p = np.sin(x_p) + np.random.randn(30) * 0.2
x_fine = np.linspace(0, 2*np.pi, 300)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(x_p, y_p, zorder=5, label='Noisy data', s=40)

colors = ['green', 'blue', 'red']
for deg, col in zip([1, 3, 9], colors):
    V = np.vander(x_p, deg+1, increasing=True)
    c, *_ = np.linalg.lstsq(V, y_p, rcond=None)
    y_pred = np.vander(x_fine, deg+1, increasing=True) @ c
    cond = np.linalg.cond(V)
    ax.plot(x_fine, y_pred, color=col, lw=2, label=f'deg={deg} (cond={cond:.1e})')

ax.set(title='Polynomial Fitting: Underfitting vs Overfitting',
       xlabel='x', ylabel='y', ylim=(-3, 3))
ax.legend(); plt.tight_layout(); plt.show()


## Summary

| Case | Rank Condition | Solution |
|---|---|---|
| Unique | $\text{rank}(A)=n$, square $A$ | `np.linalg.solve(A, b)` |
| None | $\text{rank}([A\|b]) > \text{rank}(A)$ | No solution |
| Infinite | $\text{rank}(A) < n$ | `np.linalg.lstsq(A, b)` |
| Overdetermined | $m > n$ | Least squares minimum norm |

Key insight: **least squares is the backbone of linear regression**, the simplest and most interpretable ML model.

**Next:** Chapter 4 — Vector Spaces & Orthogonality: the geometry behind why these solutions work.
